In [ ]:
%pip install -U openai fastapi uvicorn pydantic sympy requests

from pathlib import Path
Path('artifacts').mkdir(exist_ok=True, parents=True)
print('Installs done and artifacts/ created')


In [ ]:
# 🔑 Load OpenAI API key from secrets/openai_key.txt (works in notebooks AND scripts)
import os
from pathlib import Path

def _project_dir() -> Path:
    # In a script, __file__ exists; in a notebook, it doesn't.
    f = globals().get("__file__", None)
    return Path(f).resolve().parent if f else Path.cwd()

def _key_path() -> Path:
    # Allow overriding with an env var; otherwise use ./secrets/openai_key.txt
    env = os.getenv("OPENAI_KEY_PATH")
    return Path(env).expanduser().resolve() if env else (_project_dir() / "secrets" / "openai_key.txt")

def load_api_key(path: Path) -> str:
    try:
        key = path.read_text(encoding="utf-8").strip()
        if not key or not key.startswith("sk-"):
            raise ValueError("Key file is empty or not an OpenAI key.")
        return key
    except FileNotFoundError:
        raise RuntimeError(
            f"API key file not found at: {path}\n"
            "Create it and paste your key (single line). You can also set OPENAI_KEY_PATH to the file."
        )

KEY_PATH = _key_path()
OPENAI_API_KEY = load_api_key(KEY_PATH)

# Optional: quick sanity message (masked)
print(" API key loaded from:", KEY_PATH)


## Tools: search arxiv, calculate

In [26]:
# --- Tools ---
import re, requests
from sympy import sympify

USE_OFFLINE = False 
OFFLINE_ARXIV_SNIPPET = (
    'Toolformer: Language Models Can Teach Themselves to Use Tools. '
    'We show that large language models can learn to call external tools effectively.'
)

def search_arxiv(query: str) -> str:
    """
    Simulate an arXiv search or return a dummy passage for the given query.
    In a real system, this might query the arXiv API and extract a summary.
    """
    if USE_OFFLINE:
        return OFFLINE_ARXIV_SNIPPET
    url = 'http://export.arxiv.org/api/query'
    params = {'search_query': f'all:{query}', 'start': 0, 'max_results': 1}
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        text = r.text
        title_m = re.search(r'<title>(.*?)</title>', text, re.S)
        summ_m  = re.search(r'<summary>(.*?)</summary>', text, re.S)
        title = title_m.group(1).strip() if title_m else 'arXiv result'
        summary = re.sub(r'\s+', ' ', (summ_m.group(1) if summ_m else '')).strip()
        return f"{title}: {summary[:400]}"
    except Exception as e:
        return f"(arXiv lookup failed: {e})"

def calculate(expression: str) -> str:
    """
    Evaluate a mathematical expression and return the result as a string.
    """
    try:
        val = sympify(expression).evalf()
        return str(val)
    except Exception as e:
        return f"(calc error: {e})"

print('Tools ready: search_arxiv, calculate')


Tools ready: search_arxiv, calculate


## Tool registry

In [15]:
from typing import Dict, Any
import json

TOOLS: Dict[str, Any] = {
    'search_arxiv': search_arxiv,
    'calculate': calculate,
}

SYSTEM_INSTRUCTION = (
    'You are a helpful voice assistant that can call tools when useful.\n'
    'Always return STRICT JSON in one line, with keys:\n'
    '- For a tool call: {"action":"call_tool","tool":"<tool_name>","args":{...}}\n'
    '- For a direct answer: {"action":"final","answer":"<text>"}\n'
    'Do not include extra keys or commentary. Available tools:\n'
    '1) search_arxiv(query: str)\n2) calculate(expression: str)'
)

def make_user_prompt(text: str) -> str:
    return f"USER: {text}\n\nRespond with STRICT one-line JSON as specified."

print('Protocol + registry ready')


Protocol + registry ready


## LLM decision & routing

In [27]:
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = 'gpt-4o-mini'  # change if needed

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        # remove any ```...``` fence lines
        lines = [ln for ln in s.splitlines() if not ln.strip().startswith("```")]
        s = "\n".join(lines).strip()
    return s

def llm_decide(user_text: str) -> Dict[str, Any]:
    """Returns the parsed JSON dict from the model (either tool_call or final)."""
    prompt = SYSTEM_INSTRUCTION + '\n\n' + make_user_prompt(user_text)
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{'role':'user','content': prompt}],
        temperature=0.1,
    )
    raw = resp.choices[0].message.content.strip()
    raw_json = _strip_code_fences(raw)
    try:
        data = json.loads(raw_json)
    except Exception:
        data = {'action':'final','answer': raw}
    return {'raw': raw, 'data': data}

def route_llm_output(decision: Dict[str, Any]) -> Dict[str, Any]:
    """Executes tools when requested, returns a dict with full trace."""
    data = decision['data']
    trace = {'llm_raw': decision['raw'], 'tool_call': None, 'tool_result': None, 'final': None}

    if isinstance(data, dict) and data.get('action') == 'call_tool':
        tool = data.get('tool')
        args = data.get('args', {}) or {}
        fn = TOOLS.get(tool)
        if fn is None:
            trace['final'] = f'(unknown tool: {tool})'
            return trace
        try:
            result = fn(**args)
        except TypeError:
            result = fn(*list(args.values())) if args else fn('')
        trace['tool_call'] = {'tool': tool, 'args': args}
        trace['tool_result'] = result

        compose = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role':'system','content':'Write a short spoken-style reply summarizing the tool result.'},
                {'role':'user','content': f'User asked: {args}\nTool: {tool}\nResult: {result}'}
            ],
            temperature=0.3,
        )
        trace['final'] = compose.choices[0].message.content.strip()
        return trace

    trace['final'] = data.get('answer') if isinstance(data, dict) else str(data)
    return trace

print('LLM decision + router ready')


LLM decision + router ready


## FastAPI app

In [28]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title='Voice Agent with Function Calling')

class VoiceQuery(BaseModel):
    text: str

@app.post('/api/voice-query')
def voice_query(q: VoiceQuery):
    decision = llm_decide(q.text)
    trace = route_llm_output(decision)
    return {
        'query': q.text,
        'llm_raw': trace['llm_raw'],
        'tool_call': trace['tool_call'],
        'tool_result': trace['tool_result'],
        'final': trace['final'],
    }

print('FastAPI app ready (run `uvicorn app:app --reload`)')


FastAPI app ready (run `uvicorn app:app --reload`)


## Test harness

In [29]:
import json, time
LOG_PATH = 'artifacts/function_calling_testlogs.jsonl'

tests = [
    'what is 12*(5+3)?',                    # should call calculate
    'find recent papers about toolformer',  # should call search_arxiv
    'tell me a fun fact about neural nets', # should be direct answer
]

with open(LOG_PATH, 'w', encoding='utf-8') as f:
    for t in tests:
        decision = llm_decide(t)
        trace = route_llm_output(decision)
        record = {
            'timestamp': time.time(),
            'query': t,
            'llm_raw': trace['llm_raw'],
            'tool_call': trace['tool_call'],
            'tool_result': trace['tool_result'],
            'final': trace['final'],
        }
        print('\nQuery:', t)
        print('Final:', record['final'])
        print('Tool:', record['tool_call'])
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print('Saved logs to', LOG_PATH)



Query: what is 12*(5+3)?
Final: The result of the expression 12 times the sum of 5 and 3 is 96.
Tool: {'tool': 'calculate', 'args': {'expression': '12*(5+3)'}}

Query: find recent papers about toolformer
Final: The search returned a paper titled "Graph-ToolFormer," which discusses enhancing large language models (LLMs) with the ability to reason about complex graph data. It highlights that while LLMs perform well in natural language and some vision tasks, they struggle significantly with graph learning tasks. The paper proposes using prompts augmented by ChatGPT to improve this reasoning capability.
Tool: {'tool': 'search_arxiv', 'args': {'query': 'toolformer'}}

Query: tell me a fun fact about neural nets
Final: A fun fact about neural networks is that they are inspired by the human brain's structure, specifically the way neurons communicate with each other, and they can learn to recognize patterns in data, sometimes outperforming humans in tasks like image and speech recognition.
To

## Export app

In [19]:
APP_PY = '''\
from fastapi import FastAPI
from pydantic import BaseModel
from openai import OpenAI
import json
from sympy import sympify
import re, requests

def search_arxiv(query: str) -> str:
    url = 'http://export.arxiv.org/api/query'
    params = {'search_query': f'all:{query}', 'start': 0, 'max_results': 1}
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        text = r.text
        title_m = re.search(r'<title>(.*?)</title>', text, re.S)
        summ_m  = re.search(r'<summary>(.*?)</summary>', text, re.S)
        title = title_m.group(1).strip() if title_m else 'arXiv result'
        import re as _re
        summary = _re.sub(r'\\s+', ' ', (summ_m.group(1) if summ_m else '')).strip()
        return f"{title}: {summary[:400]}"
    except Exception as e:
        return f"(arXiv lookup failed: {e})"

def calculate(expression: str) -> str:
    try:
        return str(sympify(expression).evalf())
    except Exception as e:
        return f"(calc error: {e})"

TOOLS = {'search_arxiv': search_arxiv, 'calculate': calculate}
MODEL = 'gpt-4o-mini'
client = OpenAI()

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        lines = [ln for ln in s.splitlines() if not ln.strip().startswith("```")]
        s = "\\n".join(lines).strip()
    return s

def llm_decide(user_text: str):
    SYSTEM_INSTRUCTION = (
        'You are a helpful voice assistant that can call tools when useful.\\n'
        'Always return STRICT JSON in one line, with keys:\\n'
        '- For a tool call: {"action":"call_tool","tool":"<tool_name>","args":{...}}\\n'
        '- For a direct answer: {"action":"final","answer":"<text>"}'
    )
    prompt = SYSTEM_INSTRUCTION + '\\n\\nUSER: ' + user_text + '\\n\\nRespond with STRICT one-line JSON as specified.'
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{'role':'user','content': prompt}],
        temperature=0.1,
    )
    raw = resp.choices[0].message.content.strip()
    raw_json = _strip_code_fences(raw)
    try:
        data = json.loads(raw_json)
    except Exception:
        data = {'action':'final','answer': raw}
    return {'raw': raw, 'data': data}

def route_llm_output(decision):
    data = decision['data']
    trace = {'llm_raw': decision['raw'], 'tool_call': None, 'tool_result': None, 'final': None}
    if isinstance(data, dict) and data.get('action') == 'call_tool':
        tool = data.get('tool'); args = data.get('args', {}) or {}
        fn = TOOLS.get(tool)
        if fn is None:
            trace['final'] = f'(unknown tool: {tool})'
            return trace
        try:
            result = fn(**args)
        except TypeError:
            result = fn(*list(args.values())) if args else fn('')
        trace['tool_call'] = {'tool': tool, 'args': args}
        trace['tool_result'] = result
        compose = client.chat.completions.create(
            model=MODEL,
            messages=[{'role':'system','content':'Write a short spoken-style reply summarizing the tool result.'},
                     {'role':'user','content': f'User asked: {args}\\nTool: {tool}\\nResult: {result}'}],
            temperature=0.3,
        )
        trace['final'] = compose.choices[0].message.content.strip()
        return trace
    trace['final'] = data.get('answer') if isinstance(data, dict) else str(data)
    return trace

app = FastAPI(title='Voice Agent with Function Calling')
class VoiceQuery(BaseModel):
    text: str
@app.post('/api/voice-query')
def voice_query(q: VoiceQuery):
    decision = llm_decide(q.text)
    trace = route_llm_output(decision)
    return {'query': q.text, 'llm_raw': trace['llm_raw'], 'tool_call': trace['tool_call'], 'tool_result': trace['tool_result'], 'final': trace['final']}
'''
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(APP_PY)
print('Wrote app.py (run: uvicorn app:app --reload)')


Wrote app.py (run: uvicorn app:app --reload)
